In [ ]:
spark

In [ ]:
storage_account = "<Storage_account_name>"
application_id = "<App_client_ID>"
directory_id = "<Directory_ID>"

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", application_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", "<App_client_secret>")
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{directory_id}/oauth2/token")

In [ ]:
df = spark.read.format("csv")\
.option("header", "true")\
.load(f"abfss://olistdata@olistecommbrazsa.dfs.core.windows.net/bronze/olist_customers_dataset.csv")
display(df)

# Reading the Data

In [ ]:
base_path = "abfss://olistdata@olistecommbrazsa.dfs.core.windows.net/bronze/"
orders_path = base_path + "olist_orders_dataset.csv"
payments_path = base_path + "olist_order_payments_datasets.csv"
reviews_path = base_path + "olist_order_reviews_dataset.csv"
items_path = base_path + "olist_order_items_dataset.csv"
customers_path = base_path + "olist_customers_dataset.csv"
sellers_path = base_path + "olist_sellers_dataset.csv"
geolocation_path = base_path + "olist_geolocation_dataset.csv"
products_path = base_path + "olist_products_dataset.csv"


orders_df = spark.read.format("csv").option("header", "true").load(orders_path)
payments_df = spark.read.format("csv").option("header", "true").load(payments_path)
reviews_df = spark.read.format("csv").option("header", "true").load(reviews_path)
items_df = spark.read.format("csv").option("header", "true").load(items_path)
customers_df = spark.read.format("csv").option("header", "true").load(customers_path)
sellers_df = spark.read.format("csv").option("header", "true").load(sellers_path)
geolocation_df = spark.read.format("csv").option("header", "true").load(geolocation_path)
products_df = spark.read.format("csv").option("header", "true").load(products_path)

# Reading Data from Pymongo

In [ ]:
from pymongo import MongoClient

In [ ]:
# importing module
from pymongo import MongoClient

hostname = "9yqvhy.h.filess.io"
database = "olistDataNoSQL_atomicload"
port = "27018"
username = "olistDataNoSQL_atomicload"
password = "<password>"

uri = "mongodb://" + username + ":" + password + "@" + hostname + ":" + port + "/" + database

# Connect with the portnumber and host
client = MongoClient(uri)

# Access database
mydatabase = client[database]
mydatabase

In [ ]:
import pandas as pd
collection = mydatabase['product_category_name_translation']

mongo_data = pd.DataFrame(list(collection.find({})))

In [ ]:
display(products_df)

In [ ]:
mongo_data

### Cleaning the Data

In [ ]:
from pyspark.sql.functions import col, to_date, datediff, current_date

In [ ]:
def clean_dataframe(df,name):
    print("cleaning "+ name)
    return df.dropDuplicates().na.drop('all')

orders_df = clean_dataframe(orders_df, "Orders")
display(orders_df)

In [ ]:
# convert Date Columns

orders_df = orders_df.withColumn("order_purchase_timestamp", to_date(col("order_purchase_timestamp")))\
    .withColumn("order_delivered_customer_date", to_date(col("order_delivered_customer_date")))\
        .withColumn("order_estimated_delivery_date", to_date(col("order_estimated_delivery_date")))

In [ ]:
# Calculate Delivery and Time Delays

orders_df = orders_df.withColumn("actual_delivery_time", datediff("order_delivered_customer_date", "order_purchase_timestamp"))
orders_df = orders_df.withColumn("estimated_delivery_time", datediff("order_estimated_delivery_date", "order_purchase_timestamp"))
orders_df = orders_df.withColumn("delay_time", col("actual_delivery_time") - col("estimated_delivery_time"))

display(orders_df)

In [ ]:
products_df.columns

In [ ]:
# Joining Data

orders_customers_df = orders_df.join(customers_df,"customer_id","left")

orders_payments_df = orders_customers_df.join(payments_df,"order_id","left")

orders_items_df = orders_payments_df.join(items_df,"order_id","left")

orders_items_products_df = orders_items_df.join(products_df,"product_id","left")

final_df = orders_items_products_df.join(sellers_df,"seller_id","left")

In [ ]:
display(final_df)

In [ ]:
mongo_data.drop("_id", axis = 1, inplace = True)

mongo_spark_df = spark.createDataFrame(mongo_data)
display(mongo_spark_df)

In [ ]:
final_df = final_df.join(mongo_spark_df, 'product_category_name', "left")

In [ ]:
display(final_df)

In [ ]:
def remove_duplicate_columns(df):
    columns = df.columns
    
    seen_columns = set()
    columns_to_drop = []

    for column in columns:
        if column in seen_columns:
            columns_to_drop.append(column)
        else:
            seen_columns.add(column)
    
    return df.drop(*columns_to_drop)
final_df = remove_duplicate_columns(final_df)

In [ ]:
final_df.write.mode("overwrite").parquet("abfss://olistdata@olistecommbrazsa.dfs.core.windows.net/silver/")